### Understanding and Cleaning

In [ ]:
#  import libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.table import Table
from rich.box import ROUNDED
from rich.panel import Panel
from rich import box
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from rich.box import SIMPLE

In [2]:
# load the dataset
pd.set_option('display.max_columns', None)
data = pd.read_csv("work_mental_health.csv")

In [3]:
# display random sample of the dataset
data.sample(5)

,employee_id,age,gender,role,years_in_company,hours_worked_per_week,work_life_balance_score,job_satisfaction_score,stress_level_self_report,anxiety_score,depression_score,sleep_quality_score,social_support_score,presenteeism_days_per_month,burnout_risk,left_company_last_year
19732,19733,44,Male,Support,16,51,2.2,3.4,4.6,6.0,5.6,2.8,2.1,4,Moderate,0
9412,9413,53,Male,Manager,12,38,2.8,4.9,1.7,1.0,0.0,5.0,3.9,0,Low,0
3112,3113,24,Male,Manager,21,35,3.8,2.7,1.5,0.8,1.2,4.1,1.0,0,Moderate,0
12804,12805,26,Male,Executive,8,41,4.0,2.1,1.3,0.0,0.0,4.6,3.6,0,Moderate,1
9211,9212,34,Male,Developer,18,42,2.8,1.0,2.1,2.9,3.1,4.0,2.6,2,Moderate,1


In [4]:
# display the number of rows and columns in the dataset
print(f"The number of rows : {data.shape[0]}")
print(f"The number of columns : {data.shape[1]}")

The number of rows : 20000
The number of columns : 16


In [5]:
# display the information of the dataset
console = Console()
 
color_title_header = "#4D4BF4"
color_high_missing = "#86080E"
color_low_missing = "#AE4F54"
color_no_missing = "#F1EAEA"

table = Table(
    title="Data Info",
    title_style=f"bold {color_title_header}",
    box=ROUNDED,
    header_style=f"bold {color_title_header}",
    caption=f"  Memory: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

table.add_column("Column", no_wrap=True)
table.add_column("Non-Null", justify="center")
table.add_column("Unique", justify="center")
table.add_column("Dtype", justify="center")
table.add_column("Sample", justify="left", max_width=20)

for col in data.columns:
    total = len(data[col])
    non_null = data[col].count()
    null_count = total - non_null
    missing_pct = (null_count / total) * 100
    unique = data[col].nunique()
    sample = str(data[col].dropna().iloc[0])[:20] if non_null > 0 else "NaN"
    dtype = str(data[col].dtype)


    if missing_pct > 5:
        row_color = color_high_missing
    elif missing_pct > 0:
        row_color = color_low_missing
    else:
        row_color = color_no_missing

    table.add_row(
        f"[{row_color}]{col}[/{row_color}]",
        f"[{row_color}]{non_null:,}[/{row_color}]",
        f"[{row_color}]{unique:,}[/{row_color}]",
        f"[{row_color}]{dtype}[/{row_color}]",
        f"[{row_color}]{sample}[/{row_color}]"
    )

console.print(table)

                               Data Info                                
╭─────────────────────────────┬──────────┬────────┬─────────┬──────────╮
│ Column                      │ Non-Null │ Unique │  Dtype  │ Sample   │
├─────────────────────────────┼──────────┼────────┼─────────┼──────────┤
│ employee_id                 │  20,000  │ 20,000 │  int64  │ 1        │
│ age                         │  20,000  │   39   │  int64  │ 25       │
│ gender                      │  20,000  │   3    │ object  │ Female   │
│ role                        │  20,000  │   6    │ object  │ Manager  │
│ years_in_company            │  20,000  │   36   │  int64  │ 13       │
│ hours_worked_per_week       │  20,000  │   36   │  int64  │ 45       │
│ work_life_balance_score     │  20,000  │   41   │ float64 │ 4.7      │
│ job_satisfaction_score      │  20,000  │   41   │ float64 │ 2.8      │
│ stress_level_self_report    │  20,000  │   41   │ float64 │ 1.9      │
│ anxiety_score               │  20,000  │   97   │ float64 │ 2.2      │
│ depression_score            │  20,000  │   78   │ float64 │ 0.6      │
│ sleep_quality_score         │  20,000  │   38   │ float64 │ 5.0      │
│ social_support_score        │  20,000  │   41   │ float64 │ 1.1      │
│ presenteeism_days_per_month │  20,000  │   11   │  int64  │ 0        │
│ burnout_risk                │  20,000  │   3    │ object  │ Moderate │
│ left_company_last_year      │  20,000  │   2    │  int64  │ 0        │
╰─────────────────────────────┴──────────┴────────┴─────────┴──────────╯
                             Memory: 5.14 MB                            

### Overall analysis

In [6]:
console = Console()

# Age statistics panel
age_stats = data['age'].agg(['min', 'mean', 'max'])
age_stats['mean'] = age_stats['mean'].round(1)

age_table = Table(title="📈 Age Statistics", 
                  box=box.SIMPLE,
                  header_style="bold cyan")

age_table.add_column("Measure", style="yellow")
age_table.add_column("Value", style="green", justify="right")

age_table.add_row("Minimum", f"{age_stats['min']} years")
age_table.add_row("Mean", f"{age_stats['mean']} years")
age_table.add_row("Maximum", f"{age_stats['max']} years")

# Gender distribution table
gender_counts = data['gender'].value_counts()
gender_pcts = data['gender'].value_counts(normalize=True) * 100

gender_table = Table(title="📊 Gender Distribution", 
                     box=box.SIMPLE,
                     header_style="bold magenta")

gender_table.add_column("Gender", style="cyan")
gender_table.add_column("Count", style="green", justify="right")
gender_table.add_column("Percentage", style="yellow", justify="right")

for gender, count in gender_counts.items():
    gender_table.add_row(gender, f"{count:,}", f"{gender_pcts[gender]:.1f}%")

# Role distribution table
role_counts = data['role'].value_counts()
role_pcts = data['role'].value_counts(normalize=True) * 100

role_table = Table(title="💼 Role Distribution", 
                   box=box.SIMPLE,
                   header_style="bold magenta")

role_table.add_column("Role", style="cyan")
role_table.add_column("Count", style="green", justify="right")
role_table.add_column("Percentage", style="yellow", justify="right")

for role, count in role_counts.head(10).items():  # Show top 10 roles
    role_table.add_row(role, f"{count:,}", f"{role_pcts[role]:.1f}%")

# Display all tables
console.print(age_table)
console.print(gender_table)
console.print(role_table)

# Summary panel
console.print(Panel(
    f"[bold]Total Employees:[/bold] {len(data):,}\n"
    f"[bold]Age Range:[/bold] {age_stats['min']} - {age_stats['max']} years\n"
    f"[bold]Most Common Gender:[/bold] {gender_counts.index[0]}\n"
    f"[bold]Most Common Role:[/bold] {role_counts.index[0]} ({role_pcts.iloc[0]:.1f}%)",
    title="📊 Summary",
    border_style="green"
))

   📈 Age Statistics    
                        
  Measure        Value  
 ────────────────────── 
  Minimum   22.0 years  
  Mean      40.9 years  
  Maximum   60.0 years 

      📊 Gender Distribution       
                                   
  Gender       Count   Percentage  
 ───────────────────────────────── 
  Female       9,724        48.6%  
  Male         9,478        47.4%  
  Non-binary     798         4.0% 

       💼 Role Distribution       
                                  
  Role        Count   Percentage  
 ──────────────────────────────── 
  Manager     3,360        16.8%  
  Support     3,359        16.8%  
  Sales       3,355        16.8%  
  Executive   3,338        16.7%  
  HR          3,305        16.5%  
  Developer   3,283        16.4% 

╭────────────────────────────────────────────────── 📊 Summary ───────────────────────────────────────────────────╮
│ Total Employees: 20,000                                                                                         │
│ Age Range: 22.0 - 60.0 years                                                                                    │
│ Most Common Gender: Female                                                                                      │
│ Most Common Role: Manager (16.8%)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Ultra-simple console output (no libraries needed)
print("\n" + "="*60)
print("📊 WORK & BURNOUT ANALYSIS".center(60))
print("="*60)

# Work Hours
hours = data['hours_worked_per_week'].agg(['min', 'mean', 'max'])
hours['mean'] = hours['mean'].round(1)
print(f"\n⏰ WORK HOURS")
print(f"   Min: {hours['min']} hrs | Mean: {hours['mean']} hrs | Max: {hours['max']} hrs")

# Turnover
turnover = data['left_company_last_year'].value_counts(normalize=True) * 100
print(f"\n🏢 TURNOVER") 
print(f"   Stayed: {turnover.get(0, 0):.1f}% | Left: {turnover.get(1, 0):.1f}%")

# Burnout
burnout = data['burnout_risk'].value_counts(normalize=True) * 100
print(f"\n🔥 BURNOUT RISK")
print(f"   🟢 Low: {burnout.get('Low', 0):.1f}%")
print(f"   🟡 Moderate: {burnout.get('Moderate', 0):.1f}%")
print(f"   🔴 High: {burnout.get('High', 0):.1f}%")

print("\n" + "="*60)


                 📊 WORK & BURNOUT ANALYSIS                  

⏰ WORK HOURS
   Min: 35.0 hrs | Mean: 45.0 hrs | Max: 70.0 hrs

🏢 TURNOVER
   Stayed: 51.4% | Left: 48.6%

🔥 BURNOUT RISK
   🟢 Low: 18.2%
   🟡 Moderate: 70.4%
   🔴 High: 11.3%



In [8]:
# Simple version without external libraries
print("\n" + "="*60)
print("📊 MENTAL HEALTH METRICS".center(60))
print("="*60)

# Calculate stats
anxiety = data['anxiety_score'].agg(['min', 'mean', 'max'])
depression = data['depression_score'].agg(['min', 'mean', 'max'])
stress = data['stress_level_self_report'].agg(['min', 'mean', 'max'])

# Round means
anxiety['mean'] = anxiety['mean'].round(1)
depression['mean'] = depression['mean'].round(1)
stress['mean'] = stress['mean'].round(1)

# Display
print(f"\n😰 ANXIETY SCORE")
print(f"   Min: {anxiety['min']:.1f} | Mean: {anxiety['mean']:.1f} | Max: {anxiety['max']:.1f}")
print(f"   Status: {'✅ Low' if anxiety['mean'] <= 3 else '⚠️ Moderate' if anxiety['mean'] <= 6 else '🔴 High'}")

print(f"\n😔 DEPRESSION SCORE")
print(f"   Min: {depression['min']:.1f} | Mean: {depression['mean']:.1f} | Max: {depression['max']:.1f}")
print(f"   Status: {'✅ Low' if depression['mean'] <= 3 else '⚠️ Moderate' if depression['mean'] <= 6 else '🔴 High'}")

print(f"\n⚡ STRESS LEVEL")
print(f"   Min: {stress['min']:.1f} | Mean: {stress['mean']:.1f} | Max: {stress['max']:.1f}")
print(f"   Status: {'✅ Low' if stress['mean'] <= 2 else '⚠️ Moderate' if stress['mean'] <= 3.5 else '🔴 High'}")

print("\n" + "="*60)


                  📊 MENTAL HEALTH METRICS                   

😰 ANXIETY SCORE
   Min: 0.0 | Mean: 2.9 | Max: 9.8
   Status: ✅ Low

😔 DEPRESSION SCORE
   Min: 0.0 | Mean: 2.2 | Max: 8.0
   Status: ✅ Low

⚡ STRESS LEVEL
   Min: 1.0 | Mean: 2.4 | Max: 5.0
   Status: ⚠️ Moderate



In [9]:
console = Console()

# Calculate job satisfaction statistics
job_satisfaction = data['job_satisfaction_score'].agg(['min', 'mean', 'max'])
job_satisfaction['mean'] = job_satisfaction['mean'].round(1)

# Define the function BEFORE using it
def get_satisfaction_status(score):
    if score >= 4.0:
        return "✅ Very High Satisfaction"
    elif score >= 3.0:
        return "⚠️ Moderate Satisfaction"
    elif score >= 2.0:
        return "⚠️ Low Satisfaction"
    else:
        return "🔴 Very Low Satisfaction"

# Create table
table = Table(title="📊 JOB SATISFACTION SCORE ANALYSIS", 
              show_header=True,
              header_style="bold magenta",
              box=None)

table.add_column("Metric", style="bold cyan", width=20)
table.add_column("Value", justify="center", style="green", width=15)
table.add_column("Interpretation", style="yellow", width=25)

table.add_row("Minimum Score", f"{job_satisfaction['min']:.1f}", 
              "👎 Very Low" if job_satisfaction['min'] <= 2 else "👍 Acceptable")
table.add_row("Mean Score", f"{job_satisfaction['mean']:.1f}", 
              get_satisfaction_status(job_satisfaction['mean']))
table.add_row("Maximum Score", f"{job_satisfaction['max']:.1f}", 
              "🌟 Excellent" if job_satisfaction['max'] >= 4.5 else "👍 Good")

console.print(table)

# Summary panel
console.print(Panel(
    f"[bold]Job Satisfaction Score:[/bold]\n"
    f"   Range: {job_satisfaction['min']:.1f} - {job_satisfaction['max']:.1f}\n"
    f"   Average: {job_satisfaction['mean']:.1f}\n"
    f"   Status: {get_satisfaction_status(job_satisfaction['mean'])}",
    title="📈 Summary",
    border_style="cyan"
))

                📊 JOB SATISFACTION SCORE ANALYSIS                
 Metric                     Value       Interpretation            
 Minimum Score               1.0        👎 Very Low               
 Mean Score                  3.2        ⚠️ Moderate Satisfaction   
 Maximum Score               5.0        🌟 Excellent              

╭────────────────────────────────────────────────── 📈 Summary ───────────────────────────────────────────────────╮
│ Job Satisfaction Score:                                                                                         │
│    Range: 1.0 - 5.0                                                                                             │
│    Average: 3.2                                                                                                 │
│    Status: ⚠️ Moderate Satisfaction                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# Compare satisfaction with burnout risk
print("\n" + "="*60)
print("📊 JOB SATISFACTION vs BURNOUT RISK")
print("="*60)

# Satisfaction by burnout level
satisfaction_by_burnout = data.groupby('burnout_risk')['job_satisfaction_score'].mean().round(1)

for risk, score in satisfaction_by_burnout.items():
    if risk == 'High':
        print(f"🔴 {risk} Risk:  {score}")
    elif risk == 'Moderate':
        print(f"🟡 {risk} Risk: {score}")
    else:
        print(f"🟢 {risk} Risk:   {score}")

print("\n💡 Insight: Lower job satisfaction = Higher burnout risk")


📊 JOB SATISFACTION vs BURNOUT RISK
🔴 High Risk:  3.2
🟢 Low Risk:   4.4
🟡 Moderate Risk: 2.8

💡 Insight: Lower job satisfaction = Higher burnout risk


### High burnout risk

In [11]:
# Filter employees with high burnout risk
burnout_risk_high = data[data['burnout_risk'] == 'High']

In [12]:
print("="*60)
print("🔥 HIGH BURNOUT RISK ANALYSIS")
print("="*60)

# ----------------------------------------------------------------------------
# 1. Distribution by Role
# ----------------------------------------------------------------------------
print("\n📊 BY ROLE:")
print("-"*40)

# Get value counts and percentages for roles
role_counts = burnout_risk_high['role'].value_counts()
role_percentages = burnout_risk_high['role'].value_counts(normalize=True) * 100

# Create DataFrame for better display
role_distribution = pd.DataFrame({
    'Role': role_counts.index,
    'Count': role_counts.values,
    'Percentage (%)': role_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in role_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"{row['Role']:15} | {row['Count']:>4} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 2. Distribution by Gender
# ----------------------------------------------------------------------------
print("\n📊 BY GENDER:")
print("-"*40)

# Get value counts and percentages for gender
gender_counts = burnout_risk_high['gender'].value_counts()
gender_percentages = burnout_risk_high['gender'].value_counts(normalize=True) * 100

# Create DataFrame for better display
gender_distribution = pd.DataFrame({
    'Gender': gender_counts.index,
    'Count': gender_counts.values,
    'Percentage (%)': gender_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in gender_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (20 - bar_length)
    print(f"{row['Gender']:10} | {row['Count']:>4} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 3. Summary Statistics
# ----------------------------------------------------------------------------
print("\n" + "="*60)
print("📈 SUMMARY STATISTICS")
print("="*60)
print(f"Total high-risk employees: {len(burnout_risk_high):,}")
print(f"Percentage of all employees: {(len(burnout_risk_high)/len(data)*100):.2f}%")
print(f"Number of distinct roles: {burnout_risk_high['role'].nunique()}")
print(f"Most affected role: {role_distribution.iloc[0]['Role']} ({role_distribution.iloc[0]['Count']} employees)")
print(f"Gender breakdown: {gender_distribution.iloc[0]['Gender']} ({gender_distribution.iloc[0]['Percentage (%)']}%)")

print("\n" + "="*60)

🔥 HIGH BURNOUT RISK ANALYSIS

📊 BY ROLE:
----------------------------------------
Executive       |  402 ( 17.8%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Developer       |  399 ( 17.6%) | ████████░░░░░░░░░░░░░░░░░░░░░░
HR              |  375 ( 16.6%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Sales           |  373 ( 16.5%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Support         |  371 ( 16.4%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Manager         |  345 ( 15.2%) | ███████░░░░░░░░░░░░░░░░░░░░░░░

📊 BY GENDER:
----------------------------------------
Female     | 1162 ( 51.3%) | █████████████████████████
Male       | 1022 ( 45.1%) | ██████████████████████
Non-binary |   81 (  3.6%) | █░░░░░░░░░░░░░░░░░░░

📈 SUMMARY STATISTICS
Total high-risk employees: 2,265
Percentage of all employees: 11.33%
Number of distinct roles: 6
Most affected role: Executive (402 employees)
Gender breakdown: Female (51.3%)



In [13]:
# age for high burnout risk
age_stats = burnout_risk_high['age'].agg(['min', 'mean', 'max'])

# Round mean
age_stats['mean'] = age_stats['mean'].round(1)

print("\n" + "█"*50)
print(" AGE PROFILE - HIGH BURNOUT RISK")
print("█"*50)
print(f"""
  📉 Minimum Age: {age_stats['min']} years
  📊 Mean Age:    {age_stats['mean']} years
  📈 Maximum Age: {age_stats['max']} years
""")
print("█"*50 + "\n")


██████████████████████████████████████████████████
 AGE PROFILE - HIGH BURNOUT RISK
██████████████████████████████████████████████████

  📉 Minimum Age: 22.0 years
  📊 Mean Age:    40.4 years
  📈 Maximum Age: 60.0 years

██████████████████████████████████████████████████



In [14]:
# Count turnover among high-risk employees (0 = stayed, 1 = left)
turnover_counts = burnout_risk_high['left_company_last_year'].value_counts()

print("\n" + "="*60)
print("TURNOVER AMONG HIGH-RISK EMPLOYEES")
print("="*60)
print(f"\nTotal high-risk employees: {len(burnout_risk_high):,}\n")

# Display turnover statistics with formatting
for status, count in turnover_counts.items():
    pct = (count / len(burnout_risk_high)) * 100
    
    # Better label for status
    status_label = "Left Company" if status == 1 else "Stayed"
    emoji = "🚪" if status == 1 else "✅"
    
    print(f"{emoji} {status_label:<12} | {count:>6,} ({pct:>5.1f}%)")

print("\n" + "="*60)

# Calculate turnover rate
turnover_rate = (turnover_counts.get(1, 0) / len(burnout_risk_high)) * 100

# Add insight
print(f"\n💡 Insight:")
print(f"   {turnover_rate:.1f}% of high-risk employees left the company")
print(f"   This is {turnover_rate - (data['left_company_last_year'].mean() * 100):.1f}% higher than company average")

if turnover_rate > 50:
    print("🚨 CRITICAL: Most high-risk employees are leaving!")
elif turnover_rate > 30:
    print("⚠️ WARNING: High turnover among at-risk employees")
else:
    print("✅ Relatively better retention among high-risk group")

print("="*60 + "\n")


TURNOVER AMONG HIGH-RISK EMPLOYEES

Total high-risk employees: 2,265

🚪 Left Company |  1,661 ( 73.3%)
✅ Stayed       |    604 ( 26.7%)


💡 Insight:
   73.3% of high-risk employees left the company
   This is 24.8% higher than company average
🚨 CRITICAL: Most high-risk employees are leaving!



In [15]:
console = Console()

# Calculate social support statistics
social_support = burnout_risk_high['social_support_score'].agg(['min', 'mean', 'max'])
social_support['mean'] = social_support['mean'].round(1)

# Define function for status
def get_support_status(score):
    if score >= 4.0:
        return "✅ Strong Support Network"
    elif score >= 3.0:
        return "⚠️ Moderate Support"
    elif score >= 2.0:
        return "⚠️ Weak Support"
    else:
        return "🔴 Very Poor Support"

# Create table
table = Table(title="🤝 SOCIAL SUPPORT SCORE - HIGH RISK EMPLOYEES", 
              show_header=True,
              header_style="bold magenta",
              box=None)

table.add_column("Metric", style="bold cyan", width=20)
table.add_column("Value", justify="center", style="green", width=15)
table.add_column("Interpretation", style="yellow", width=25)

table.add_row("Minimum Score", f"{social_support['min']:.1f}", 
              get_support_status(social_support['min']))
table.add_row("Mean Score", f"{social_support['mean']:.1f}", 
              get_support_status(social_support['mean']))
table.add_row("Maximum Score", f"{social_support['max']:.1f}", 
              get_support_status(social_support['max']))

console.print(table)

# Summary panel
console.print(Panel(
    f"[bold]Social Support Score (High-Risk Employees):[/bold]\n"
    f"   Range: {social_support['min']:.1f} - {social_support['max']:.1f}\n"
    f"   Average: {social_support['mean']:.1f}\n"
    f"   Status: {get_support_status(social_support['mean'])}",
    title="📈 Summary",
    border_style="cyan"
))


          🤝 SOCIAL SUPPORT SCORE - HIGH RISK EMPLOYEES           
 Metric                     Value       Interpretation            
 Minimum Score               1.0        🔴 Very Poor Support      
 Mean Score                  3.0        ⚠️ Moderate Support        
 Maximum Score               5.0        ✅ Strong Support Network 

╭────────────────────────────────────────────────── 📈 Summary ───────────────────────────────────────────────────╮
│ Social Support Score (High-Risk Employees):                                                                     │
│    Range: 1.0 - 5.0                                                                                             │
│    Average: 3.0                                                                                                 │
│    Status: ⚠️ Moderate Support                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
# Additional analysis by tenure groups
print("\n" + "="*60)
print("📊 BURNOUT RISK BY TENURE CATEGORY")
print("="*60)

# Define tenure categories
def tenure_category(years):
    if years < 2:
        return "0-2 years (New)"
    elif years < 5:
        return "2-5 years"
    elif years < 10:
        return "5-10 years"
    elif years < 15:
        return "10-15 years"
    else:
        return "15+ years"

# Calculate burnout risk by tenure
tenure_risk = data.groupby(data['years_in_company'].apply(tenure_category))['burnout_risk'].apply(
    lambda x: (x == 'High').mean() * 100
).round(1)

for tenure, risk_pct in tenure_risk.sort_values().items():
    bar = "█" * int(risk_pct / 2)
    emoji = "🔴" if risk_pct > 20 else "🟡" if risk_pct > 10 else "🟢"
    print(f"{emoji} {tenure:<15} | {risk_pct:>5.1f}% high-risk | {bar}")

print("="*60)


📊 BURNOUT RISK BY TENURE CATEGORY
🟡 10-15 years     |  10.5% high-risk | █████
🟡 5-10 years      |  10.7% high-risk | █████
🟡 2-5 years       |  11.5% high-risk | █████
🟡 15+ years       |  11.5% high-risk | █████
🟡 0-2 years (New) |  12.7% high-risk | ██████


In [17]:
# Distribution of work hours among high-risk employees
print("\n" + "="*60)
print("📊 WORK HOURS DISTRIBUTION (High-Risk Employees)")
print("="*60)

# Define hour categories
hour_categories = {
    '≤40 hrs (Healthy)': 0,
    '41-45 hrs (Borderline)': 0,
    '46-50 hrs (Warning)': 0,
    '51-60 hrs (Critical)': 0,
    '60+ hrs (Severe)': 0
}

for hours in burnout_risk_high['hours_worked_per_week']:
    if hours <= 40:
        hour_categories['≤40 hrs (Healthy)'] += 1
    elif hours <= 45:
        hour_categories['41-45 hrs (Borderline)'] += 1
    elif hours <= 50:
        hour_categories['46-50 hrs (Warning)'] += 1
    elif hours <= 60:
        hour_categories['51-60 hrs (Critical)'] += 1
    else:
        hour_categories['60+ hrs (Severe)'] += 1

for category, count in hour_categories.items():
    pct = (count / len(burnout_risk_high)) * 100
    bar = "█" * int(pct / 2)
    emoji = "🔴" if "Critical" in category or "Severe" in category else "🟡" if "Warning" in category else "🟢"
    print(f"{emoji} {category:<22} | {count:>5,} ({pct:>5.1f}%) | {bar}")

print("="*60)


📊 WORK HOURS DISTRIBUTION (High-Risk Employees)
🟢 ≤40 hrs (Healthy)      |    18 (  0.8%) | 
🟢 41-45 hrs (Borderline) |    89 (  3.9%) | █
🟡 46-50 hrs (Warning)    |   200 (  8.8%) | ████
🔴 51-60 hrs (Critical)   | 1,487 ( 65.7%) | ████████████████████████████████
🔴 60+ hrs (Severe)       |   471 ( 20.8%) | ██████████


### Moderate burnout risk


In [18]:
# Filter employees with Moderate burnout risk
burnout_risk_moderate = data[data['burnout_risk'] == 'Moderate']

In [19]:
console = Console()

print("="*60)
print("🟡 MODERATE BURNOUT RISK ANALYSIS")
print("="*60)

# ----------------------------------------------------------------------------
# 1. Distribution by Role
# ----------------------------------------------------------------------------
print("\n📊 BY ROLE:")
print("-"*40)

# Get value counts and percentages for roles
role_counts = burnout_risk_moderate['role'].value_counts()
role_percentages = burnout_risk_moderate['role'].value_counts(normalize=True) * 100

# Create DataFrame for better display
role_distribution = pd.DataFrame({
    'Role': role_counts.index,
    'Count': role_counts.values,
    'Percentage (%)': role_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in role_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"{row['Role']:15} | {row['Count']:>5,} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 2. Distribution by Gender
# ----------------------------------------------------------------------------
print("\n📊 BY GENDER:")
print("-"*40)

# Get value counts and percentages for gender
gender_counts = burnout_risk_moderate['gender'].value_counts()
gender_percentages = burnout_risk_moderate['gender'].value_counts(normalize=True) * 100

# Create DataFrame for better display
gender_distribution = pd.DataFrame({
    'Gender': gender_counts.index,
    'Count': gender_counts.values,
    'Percentage (%)': gender_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in gender_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (20 - bar_length)
    print(f"{row['Gender']:10} | {row['Count']:>5,} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 3. Summary Statistics
# ----------------------------------------------------------------------------
print("\n" + "="*60)
print("📈 SUMMARY STATISTICS")
print("="*60)
print(f"Total moderate-risk employees: {len(burnout_risk_moderate):,}")
print(f"Percentage of all employees: {(len(burnout_risk_moderate)/len(data)*100):.2f}%")
print(f"Number of distinct roles: {burnout_risk_moderate['role'].nunique()}")
print(f"Most affected role: {role_distribution.iloc[0]['Role']} ({role_distribution.iloc[0]['Count']:,} employees)")
print(f"Gender breakdown: {gender_distribution.iloc[0]['Gender']} ({gender_distribution.iloc[0]['Percentage (%)']}%)")

print("\n" + "="*60)

🟡 MODERATE BURNOUT RISK ANALYSIS

📊 BY ROLE:
----------------------------------------
Support         | 2,390 ( 17.0%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Manager         | 2,381 ( 16.9%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Sales           | 2,381 ( 16.9%) | ████████░░░░░░░░░░░░░░░░░░░░░░
HR              | 2,338 ( 16.6%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Executive       | 2,309 ( 16.4%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Developer       | 2,286 ( 16.2%) | ████████░░░░░░░░░░░░░░░░░░░░░░

📊 BY GENDER:
----------------------------------------
Female     | 6,760 ( 48.0%) | ███████████████████████
Male       | 6,755 ( 48.0%) | ███████████████████████
Non-binary |   570 (  4.0%) | ██░░░░░░░░░░░░░░░░░░

📈 SUMMARY STATISTICS
Total moderate-risk employees: 14,085
Percentage of all employees: 70.43%
Number of distinct roles: 6
Most affected role: Support (2,390 employees)
Gender breakdown: Female (47.99%)



In [20]:
# Age statistics for moderate burnout risk
age_stats_moderate = burnout_risk_moderate['age'].agg(['min', 'mean', 'max'])

# Round mean to 1 decimal
age_stats_moderate['mean'] = age_stats_moderate['mean'].round(1)

print("\n" + "█"*50)
print(" 🟡 AGE PROFILE - MODERATE BURNOUT RISK")
print("█"*50)
print(f"""
  📉 Minimum Age: {age_stats_moderate['min']} years
  📊 Mean Age:    {age_stats_moderate['mean']} years
  📈 Maximum Age: {age_stats_moderate['max']} years
""")
print("█"*50 + "\n")


██████████████████████████████████████████████████
 🟡 AGE PROFILE - MODERATE BURNOUT RISK
██████████████████████████████████████████████████

  📉 Minimum Age: 22.0 years
  📊 Mean Age:    40.9 years
  📈 Maximum Age: 60.0 years

██████████████████████████████████████████████████



In [21]:
# Count turnover among moderate-risk employees (0 = stayed, 1 = left)
turnover_counts_moderate = burnout_risk_moderate['left_company_last_year'].value_counts()

print("\n" + "="*60)
print("🟡 TURNOVER AMONG MODERATE-RISK EMPLOYEES")
print("="*60)
print(f"\nTotal moderate-risk employees: {len(burnout_risk_moderate):,}\n")

# Display turnover statistics with formatting
for status, count in turnover_counts_moderate.items():
    pct = (count / len(burnout_risk_moderate)) * 100
    
    # Better label for status
    status_label = "Left Company" if status == 1 else "Stayed"
    emoji = "🚪" if status == 1 else "✅"
    
    print(f"{emoji} {status_label:<12} | {count:>6,} ({pct:>5.1f}%)")

print("\n" + "="*60)

# Calculate turnover rate
turnover_rate_moderate = (turnover_counts_moderate.get(1, 0) / len(burnout_risk_moderate)) * 100
turnover_rate_overall = data['left_company_last_year'].mean() * 100

# Add insight
print(f"\n💡 Insight:")
print(f"   {turnover_rate_moderate:.1f}% of moderate-risk employees left the company")
print(f"   This is {turnover_rate_moderate - turnover_rate_overall:.1f}% {'higher' if turnover_rate_moderate > turnover_rate_overall else 'lower'} than company average ({turnover_rate_overall:.1f}%)")

if turnover_rate_moderate > 50:
    print("🚨 CRITICAL: Most moderate-risk employees are leaving!")
elif turnover_rate_moderate > 30:
    print("⚠️ WARNING: High turnover among moderate-risk employees")
elif turnover_rate_moderate > 20:
    print("⚠️ CAUTION: Elevated turnover rate in this group")
else:
    print("✅ Good retention among moderate-risk group")

print("="*60 + "\n")


🟡 TURNOVER AMONG MODERATE-RISK EMPLOYEES

Total moderate-risk employees: 14,085

🚪 Left Company |  7,619 ( 54.1%)
✅ Stayed       |  6,466 ( 45.9%)


💡 Insight:
   54.1% of moderate-risk employees left the company
   This is 5.5% higher than company average (48.6%)
🚨 CRITICAL: Most moderate-risk employees are leaving!



In [22]:
console = Console()

# Calculate social support statistics
social_support_moderate = burnout_risk_moderate['social_support_score'].agg(['min', 'mean', 'max'])
social_support_moderate['mean'] = social_support_moderate['mean'].round(1)

# Define function for status
def get_support_status(score):
    if score >= 4.0:
        return "✅ Strong Support Network"
    elif score >= 3.0:
        return "⚠️ Moderate Support"
    elif score >= 2.0:
        return "⚠️ Weak Support"
    else:
        return "🔴 Very Poor Support"

# Create table
table = Table(title="🤝 SOCIAL SUPPORT SCORE - MODERATE RISK EMPLOYEES", 
              show_header=True,
              header_style="bold magenta",
              box=None)

table.add_column("Metric", style="bold cyan", width=20)
table.add_column("Value", justify="center", style="green", width=15)
table.add_column("Interpretation", style="yellow", width=25)

table.add_row("Minimum Score", f"{social_support_moderate['min']:.1f}", 
              get_support_status(social_support_moderate['min']))
table.add_row("Mean Score", f"{social_support_moderate['mean']:.1f}", 
              get_support_status(social_support_moderate['mean']))
table.add_row("Maximum Score", f"{social_support_moderate['max']:.1f}", 
              get_support_status(social_support_moderate['max']))

console.print(table)

# Summary panel
console.print(Panel(
    f"[bold]Social Support Score (Moderate-Risk Employees):[/bold]\n"
    f"   Range: {social_support_moderate['min']:.1f} - {social_support_moderate['max']:.1f}\n"
    f"   Average: {social_support_moderate['mean']:.1f}\n"
    f"   Status: {get_support_status(social_support_moderate['mean'])}",
    title="📈 Summary",
    border_style="yellow"
))

        🤝 SOCIAL SUPPORT SCORE - MODERATE RISK EMPLOYEES         
 Metric                     Value       Interpretation            
 Minimum Score               1.0        🔴 Very Poor Support      
 Mean Score                  3.0        ⚠️ Moderate Support        
 Maximum Score               5.0        ✅ Strong Support Network 

╭────────────────────────────────────────────────── 📈 Summary ───────────────────────────────────────────────────╮
│ Social Support Score (Moderate-Risk Employees):                                                                 │
│    Range: 1.0 - 5.0                                                                                             │
│    Average: 3.0                                                                                                 │
│    Status: ⚠️ Moderate Support                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# Additional analysis by tenure groups for MODERATE risk
print("\n" + "="*60)
print("📊 MODERATE BURNOUT RISK BY TENURE CATEGORY")
print("="*60)

# Define tenure categories
def tenure_category(years):
    if years < 2:
        return "0-2 years (New)"
    elif years < 5:
        return "2-5 years"
    elif years < 10:
        return "5-10 years"
    elif years < 15:
        return "10-15 years"
    else:
        return "15+ years"

# Calculate MODERATE burnout risk by tenure
tenure_moderate_risk = data.groupby(data['years_in_company'].apply(tenure_category))['burnout_risk'].apply(
    lambda x: (x == 'Moderate').mean() * 100
).round(1)

for tenure, risk_pct in tenure_moderate_risk.sort_values().items():
    bar = "█" * int(risk_pct / 2)
    # Different emoji for moderate risk
    emoji = "🟡" if risk_pct > 50 else "🟢" if risk_pct > 30 else "🔵"
    print(f"{emoji} {tenure:<15} | {risk_pct:>5.1f}% moderate-risk | {bar}")

print("="*60)


📊 MODERATE BURNOUT RISK BY TENURE CATEGORY
🟡 0-2 years (New) |  67.5% moderate-risk | █████████████████████████████████
🟡 5-10 years      |  69.3% moderate-risk | ██████████████████████████████████
🟡 15+ years       |  70.5% moderate-risk | ███████████████████████████████████
🟡 2-5 years       |  70.8% moderate-risk | ███████████████████████████████████
🟡 10-15 years     |  72.3% moderate-risk | ████████████████████████████████████


In [24]:
print("\n" + "="*60)
print("📊 WORK HOURS DISTRIBUTION (Moderate-Risk Employees)")
print("="*60)

# Define hour categories
hour_categories = {
    '≤40 hrs (Healthy)': 0,
    '41-45 hrs (Borderline)': 0,
    '46-50 hrs (Warning)': 0,
    '51-60 hrs (Critical)': 0,
    '60+ hrs (Severe)': 0
}

for hours in burnout_risk_moderate['hours_worked_per_week']:
    if hours <= 40:
        hour_categories['≤40 hrs (Healthy)'] += 1
    elif hours <= 45:
        hour_categories['41-45 hrs (Borderline)'] += 1
    elif hours <= 50:
        hour_categories['46-50 hrs (Warning)'] += 1
    elif hours <= 60:
        hour_categories['51-60 hrs (Critical)'] += 1
    else:
        hour_categories['60+ hrs (Severe)'] += 1

for category, count in hour_categories.items():
    pct = (count / len(burnout_risk_moderate)) * 100
    bar = "█" * int(pct / 2)
    # Different emoji for moderate risk (yellow)
    emoji = "🔴" if "Critical" in category or "Severe" in category else "🟡" if "Warning" in category else "🟢"
    print(f"{emoji} {category:<22} | {count:>6,} ({pct:>5.1f}%) | {bar}")

print("="*60)


📊 WORK HOURS DISTRIBUTION (Moderate-Risk Employees)
🟢 ≤40 hrs (Healthy)      |  4,014 ( 28.5%) | ██████████████
🟢 41-45 hrs (Borderline) |  3,272 ( 23.2%) | ███████████
🟡 46-50 hrs (Warning)    |  4,247 ( 30.2%) | ███████████████
🔴 51-60 hrs (Critical)   |  2,552 ( 18.1%) | █████████
🔴 60+ hrs (Severe)       |      0 (  0.0%) | 


### Low burnout risk

In [25]:
# Filter employees with Low burnout risk
burnout_risk_moderate = data[data['burnout_risk'] == 'Low']

In [26]:
console = Console()

# Filter employees with Low burnout risk
burnout_risk_low = data[data['burnout_risk'] == 'Low']

print("="*60)
print("🟢 LOW BURNOUT RISK ANALYSIS")
print("="*60)

# ----------------------------------------------------------------------------
# 1. Distribution by Role
# ----------------------------------------------------------------------------
print("\n📊 BY ROLE:")
print("-"*40)

# Get value counts and percentages for roles
role_counts = burnout_risk_low['role'].value_counts()
role_percentages = burnout_risk_low['role'].value_counts(normalize=True) * 100

# Create DataFrame for better display
role_distribution = pd.DataFrame({
    'Role': role_counts.index,
    'Count': role_counts.values,
    'Percentage (%)': role_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in role_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"{row['Role']:15} | {row['Count']:>5,} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 2. Distribution by Gender
# ----------------------------------------------------------------------------
print("\n📊 BY GENDER:")
print("-"*40)

# Get value counts and percentages for gender
gender_counts = burnout_risk_low['gender'].value_counts()
gender_percentages = burnout_risk_low['gender'].value_counts(normalize=True) * 100

# Create DataFrame for better display
gender_distribution = pd.DataFrame({
    'Gender': gender_counts.index,
    'Count': gender_counts.values,
    'Percentage (%)': gender_percentages.values.round(2)
}).sort_values('Count', ascending=False)

# Display with visual bars
for _, row in gender_distribution.iterrows():
    bar_length = int(row['Percentage (%)'] / 2)
    bar = "█" * bar_length + "░" * (20 - bar_length)
    print(f"{row['Gender']:10} | {row['Count']:>5,} ({row['Percentage (%)']:>5.1f}%) | {bar}")

# ----------------------------------------------------------------------------
# 3. Summary Statistics
# ----------------------------------------------------------------------------
print("\n" + "="*60)
print("📈 SUMMARY STATISTICS")
print("="*60)
print(f"Total low-risk employees: {len(burnout_risk_low):,}")
print(f"Percentage of all employees: {(len(burnout_risk_low)/len(data)*100):.2f}%")
print(f"Number of distinct roles: {burnout_risk_low['role'].nunique()}")
print(f"Most common role: {role_distribution.iloc[0]['Role']} ({role_distribution.iloc[0]['Count']:,} employees)")
print(f"Gender breakdown: {gender_distribution.iloc[0]['Gender']} ({gender_distribution.iloc[0]['Percentage (%)']}%)")

print("\n" + "="*60)

🟢 LOW BURNOUT RISK ANALYSIS

📊 BY ROLE:
----------------------------------------
Manager         |   634 ( 17.4%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Executive       |   627 ( 17.2%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Sales           |   601 ( 16.5%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Developer       |   598 ( 16.4%) | ████████░░░░░░░░░░░░░░░░░░░░░░
Support         |   598 ( 16.4%) | ████████░░░░░░░░░░░░░░░░░░░░░░
HR              |   592 ( 16.2%) | ████████░░░░░░░░░░░░░░░░░░░░░░

📊 BY GENDER:
----------------------------------------
Female     | 1,802 ( 49.4%) | ████████████████████████
Male       | 1,701 ( 46.6%) | ███████████████████████
Non-binary |   147 (  4.0%) | ██░░░░░░░░░░░░░░░░░░

📈 SUMMARY STATISTICS
Total low-risk employees: 3,650
Percentage of all employees: 18.25%
Number of distinct roles: 6
Most common role: Manager (634 employees)
Gender breakdown: Female (49.37%)



In [27]:
# Age statistics for low burnout risk
age_stats_low = burnout_risk_low['age'].agg(['min', 'mean', 'max'])

# Round mean to 1 decimal
age_stats_low['mean'] = age_stats_low['mean'].round(1)

print("\n" + "█"*50)
print(" 🟢 AGE PROFILE - LOW BURNOUT RISK")
print("█"*50)
print(f"""
  📉 Minimum Age: {age_stats_low['min']} years
  📊 Mean Age:    {age_stats_low['mean']} years
  📈 Maximum Age: {age_stats_low['max']} years
""")
print("█"*50 + "\n")


██████████████████████████████████████████████████
 🟢 AGE PROFILE - LOW BURNOUT RISK
██████████████████████████████████████████████████

  📉 Minimum Age: 22.0 years
  📊 Mean Age:    40.9 years
  📈 Maximum Age: 60.0 years

██████████████████████████████████████████████████



In [28]:
# Count turnover among low-risk employees (0 = stayed, 1 = left)
turnover_counts_low = burnout_risk_low['left_company_last_year'].value_counts()

print("\n" + "="*60)
print("🟢 TURNOVER AMONG LOW-RISK EMPLOYEES")
print("="*60)
print(f"\nTotal low-risk employees: {len(burnout_risk_low):,}\n")

# Display turnover statistics with formatting
for status, count in turnover_counts_low.items():
    pct = (count / len(burnout_risk_low)) * 100
    
    # Better label for status
    status_label = "Left Company" if status == 1 else "Stayed"
    emoji = "🚪" if status == 1 else "✅"
    
    print(f"{emoji} {status_label:<12} | {count:>6,} ({pct:>5.1f}%)")

print("\n" + "="*60)

# Calculate turnover rate
turnover_rate_low = (turnover_counts_low.get(1, 0) / len(burnout_risk_low)) * 100
turnover_rate_overall = data['left_company_last_year'].mean() * 100

# Add insight
print(f"\n💡 Insight:")
print(f"   {turnover_rate_low:.1f}% of low-risk employees left the company")
print(f"   This is {abs(turnover_rate_low - turnover_rate_overall):.1f}% {'higher' if turnover_rate_low > turnover_rate_overall else 'lower'} than company average ({turnover_rate_overall:.1f}%)")

if turnover_rate_low > 50:
    print("🚨 CRITICAL: Most low-risk employees are leaving!")
elif turnover_rate_low > 30:
    print("⚠️ WARNING: High turnover among low-risk employees")
elif turnover_rate_low > 20:
    print("⚠️ CAUTION: Elevated turnover rate in this group")
else:
    print("✅ Excellent retention among low-risk group - This is a benchmark!")

print("="*60 + "\n")


🟢 TURNOVER AMONG LOW-RISK EMPLOYEES

Total low-risk employees: 3,650

✅ Stayed       |  3,217 ( 88.1%)
🚪 Left Company |    433 ( 11.9%)


💡 Insight:
   11.9% of low-risk employees left the company
   This is 36.7% lower than company average (48.6%)
✅ Excellent retention among low-risk group - This is a benchmark!



In [29]:
console = Console()

# Filter low-risk employees
burnout_risk_low = data[data['burnout_risk'] == 'Low']

# Calculate social support statistics
social_support_low = burnout_risk_low['social_support_score'].agg(['min', 'mean', 'max'])
social_support_low['mean'] = social_support_low['mean'].round(1)

# Define function for status
def get_support_status(score):
    if score >= 4.0:
        return "✅ Strong Support Network"
    elif score >= 3.0:
        return "⚠️ Moderate Support"
    elif score >= 2.0:
        return "⚠️ Weak Support"
    else:
        return "🔴 Very Poor Support"

# Create table
table = Table(title="🤝 SOCIAL SUPPORT SCORE - LOW RISK EMPLOYEES", 
              show_header=True,
              header_style="bold green",
              box=None)

table.add_column("Metric", style="bold cyan", width=20)
table.add_column("Value", justify="center", style="green", width=15)
table.add_column("Interpretation", style="yellow", width=25)

table.add_row("Minimum Score", f"{social_support_low['min']:.1f}", 
              get_support_status(social_support_low['min']))
table.add_row("Mean Score", f"{social_support_low['mean']:.1f}", 
              get_support_status(social_support_low['mean']))
table.add_row("Maximum Score", f"{social_support_low['max']:.1f}", 
              get_support_status(social_support_low['max']))

console.print(table)

# Summary panel
console.print(Panel(
    f"[bold]Social Support Score (Low-Risk Employees):[/bold]\n"
    f"   Range: {social_support_low['min']:.1f} - {social_support_low['max']:.1f}\n"
    f"   Average: {social_support_low['mean']:.1f}\n"
    f"   Status: {get_support_status(social_support_low['mean'])}",
    title="📈 Summary",
    border_style="green"
))

           🤝 SOCIAL SUPPORT SCORE - LOW RISK EMPLOYEES           
 Metric                     Value       Interpretation            
 Minimum Score               1.0        🔴 Very Poor Support      
 Mean Score                  3.0        ⚠️ Moderate Support        
 Maximum Score               5.0        ✅ Strong Support Network 

╭────────────────────────────────────────────────── 📈 Summary ───────────────────────────────────────────────────╮
│ Social Support Score (Low-Risk Employees):                                                                      │
│    Range: 1.0 - 5.0                                                                                             │
│    Average: 3.0                                                                                                 │
│    Status: ⚠️ Moderate Support                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
# Additional analysis by tenure groups for LOW risk
print("\n" + "="*60)
print("📊 LOW BURNOUT RISK BY TENURE CATEGORY")
print("="*60)

# Define tenure categories
def tenure_category(years):
    if years < 2:
        return "0-2 years (New)"
    elif years < 5:
        return "2-5 years"
    elif years < 10:
        return "5-10 years"
    elif years < 15:
        return "10-15 years"
    else:
        return "15+ years"

# Calculate LOW burnout risk by tenure
tenure_low_risk = data.groupby(data['years_in_company'].apply(tenure_category))['burnout_risk'].apply(
    lambda x: (x == 'Low').mean() * 100
).round(1)

for tenure, risk_pct in tenure_low_risk.sort_values(ascending=False).items():
    bar = "█" * int(risk_pct / 2)
    # Different emoji for low risk (green)
    emoji = "🟢" if risk_pct > 25 else "🟡" if risk_pct > 15 else "🔴"
    print(f"{emoji} {tenure:<15} | {risk_pct:>5.1f}% low-risk | {bar}")

print("="*60)


📊 LOW BURNOUT RISK BY TENURE CATEGORY
🟡 5-10 years      |  20.0% low-risk | ██████████
🟡 0-2 years (New) |  19.8% low-risk | █████████
🟡 15+ years       |  18.0% low-risk | █████████
🟡 2-5 years       |  17.7% low-risk | ████████
🟡 10-15 years     |  17.3% low-risk | ████████


In [31]:
# Filter low-risk employees
burnout_risk_low = data[data['burnout_risk'] == 'Low']

print("\n" + "="*60)
print("📊 WORK HOURS DISTRIBUTION (Low-Risk Employees)")
print("="*60)

# Define hour categories
hour_categories = {
    '≤40 hrs (Healthy)': 0,
    '41-45 hrs (Borderline)': 0,
    '46-50 hrs (Warning)': 0,
    '51-60 hrs (Critical)': 0,
    '60+ hrs (Severe)': 0
}

for hours in burnout_risk_low['hours_worked_per_week']:
    if hours <= 40:
        hour_categories['≤40 hrs (Healthy)'] += 1
    elif hours <= 45:
        hour_categories['41-45 hrs (Borderline)'] += 1
    elif hours <= 50:
        hour_categories['46-50 hrs (Warning)'] += 1
    elif hours <= 60:
        hour_categories['51-60 hrs (Critical)'] += 1
    else:
        hour_categories['60+ hrs (Severe)'] += 1

for category, count in hour_categories.items():
    pct = (count / len(burnout_risk_low)) * 100
    bar = "█" * int(pct / 2)
    # Emoji for low risk (green for healthy, yellow for borderline/warning)
    emoji = "🔴" if "Critical" in category or "Severe" in category else "🟡" if "Warning" in category else "🟢"
    print(f"{emoji} {category:<22} | {count:>6,} ({pct:>5.1f}%) | {bar}")

print("="*60)


📊 WORK HOURS DISTRIBUTION (Low-Risk Employees)
🟢 ≤40 hrs (Healthy)      |  2,169 ( 59.4%) | █████████████████████████████
🟢 41-45 hrs (Borderline) |  1,481 ( 40.6%) | ████████████████████
🟡 46-50 hrs (Warning)    |      0 (  0.0%) | 
🔴 51-60 hrs (Critical)   |      0 (  0.0%) | 
🔴 60+ hrs (Severe)       |      0 (  0.0%) | 


 
### Comprehensive comparison: low vs moderate vs high burnout risk

In [32]:
def save_analysis_to_file(data, filename='burnout_analysis_report.txt'):
    """
    Save burnout risk analysis to a text file
    """
    import sys
    from datetime import datetime
    
    # Open file for writing
    with open(filename, 'w', encoding='utf-8') as f:
        # Write header
        f.write("="*70 + "\n")
        f.write("BURNOUT RISK ANALYSIS REPORT\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")
        
        # Save original stdout
        original_stdout = sys.stdout
        
        # Redirect stdout to file
        sys.stdout = f
        
        # Call your analysis function (or paste code here)
        # ========== ANALYSIS CODE ==========
        
        # Filter risk groups
        burnout_risk_low = data[data['burnout_risk'] == 'Low']
        burnout_risk_moderate = data[data['burnout_risk'] == 'Moderate']
        burnout_risk_high = data[data['burnout_risk'] == 'High']
        
        # Population overview
        print("📊 POPULATION OVERVIEW")
        print("-"*50)
        total = len(data)
        low_count = len(burnout_risk_low)
        moderate_count = len(burnout_risk_moderate)
        high_count = len(burnout_risk_high)
        
        print(f"  🟢 Low Risk      : {low_count:>6,} employees ({low_count/total*100:>5.1f}%)")
        print(f"  🟡 Moderate Risk : {moderate_count:>6,} employees ({moderate_count/total*100:>5.1f}%)")
        print(f"  🔴 High Risk     : {high_count:>6,} employees ({high_count/total*100:>5.1f}%)")
        print(f"  {'─'*50}")
        print(f"  📊 Total         : {total:>6,} employees (100.0%)")
        
        # Age comparison
        print("\n📊 AGE PROFILE")
        print("-"*50)
        
        def get_age_stats(df, name):
            return {
                'name': name,
                'min': df['age'].min(),
                'mean': df['age'].mean().round(1),
                'max': df['age'].max()
            }
        
        low_age = get_age_stats(burnout_risk_low, "🟢 Low")
        moderate_age = get_age_stats(burnout_risk_moderate, "🟡 Moderate")
        high_age = get_age_stats(burnout_risk_high, "🔴 High")
        
        print(f"  {'Risk Level':<14} {'Min':>6} {'Mean':>8} {'Max':>6}")
        print(f"  {'-'*40}")
        print(f"  {low_age['name']:<14} {low_age['min']:>6} {low_age['mean']:>8.1f} {low_age['max']:>6}")
        print(f"  {moderate_age['name']:<14} {moderate_age['min']:>6} {moderate_age['mean']:>8.1f} {moderate_age['max']:>6}")
        print(f"  {high_age['name']:<14} {high_age['min']:>6} {high_age['mean']:>8.1f} {high_age['max']:>6}")
        
        # Work hours comparison
        print("\n📊 WORK HOURS PER WEEK")
        print("-"*50)
        
        def get_hours_stats(df, name):
            return {
                'name': name,
                'min': df['hours_worked_per_week'].min(),
                'mean': df['hours_worked_per_week'].mean().round(1),
                'max': df['hours_worked_per_week'].max()
            }
        
        low_hours = get_hours_stats(burnout_risk_low, "🟢 Low")
        moderate_hours = get_hours_stats(burnout_risk_moderate, "🟡 Moderate")
        high_hours = get_hours_stats(burnout_risk_high, "🔴 High")
        
        print(f"  {'Risk Level':<14} {'Min':>6} {'Mean':>8} {'Max':>6}")
        print(f"  {'-'*40}")
        print(f"  {low_hours['name']:<14} {low_hours['min']:>6} {low_hours['mean']:>8.1f} {low_hours['max']:>6}")
        print(f"  {moderate_hours['name']:<14} {moderate_hours['min']:>6} {moderate_hours['mean']:>8.1f} {moderate_hours['max']:>6}")
        print(f"  {high_hours['name']:<14} {high_hours['min']:>6} {high_hours['mean']:>8.1f} {high_hours['max']:>6}")
        
        # Social support comparison
        print("\n📊 SOCIAL SUPPORT SCORE")
        print("-"*50)
        
        def get_support_stats(df, name):
            return {
                'name': name,
                'min': df['social_support_score'].min(),
                'mean': df['social_support_score'].mean().round(1),
                'max': df['social_support_score'].max()
            }
        
        low_support = get_support_stats(burnout_risk_low, "🟢 Low")
        moderate_support = get_support_stats(burnout_risk_moderate, "🟡 Moderate")
        high_support = get_support_stats(burnout_risk_high, "🔴 High")
        
        print(f"  {'Risk Level':<14} {'Min':>6} {'Mean':>8} {'Max':>6}")
        print(f"  {'-'*40}")
        print(f"  {low_support['name']:<14} {low_support['min']:>6.1f} {low_support['mean']:>8.1f} {low_support['max']:>6.1f}")
        print(f"  {moderate_support['name']:<14} {moderate_support['min']:>6.1f} {moderate_support['mean']:>8.1f} {moderate_support['max']:>6.1f}")
        print(f"  {high_support['name']:<14} {high_support['min']:>6.1f} {high_support['mean']:>8.1f} {high_support['max']:>6.1f}")
        
        # Turnover comparison
        print("\n📊 TURNOVER RATE")
        print("-"*50)
        
        def get_turnover_stats(df, name):
            total = len(df)
            left = df['left_company_last_year'].sum()
            rate = (left / total) * 100 if total > 0 else 0
            return {'name': name, 'left': left, 'total': total, 'rate': rate}
        
        low_turn = get_turnover_stats(burnout_risk_low, "🟢 Low")
        moderate_turn = get_turnover_stats(burnout_risk_moderate, "🟡 Moderate")
        high_turn = get_turnover_stats(burnout_risk_high, "🔴 High")
        overall_rate = (data['left_company_last_year'].sum() / len(data)) * 100
        
        print(f"  {'Risk Level':<14} {'Left':>8} {'Total':>8} {'Rate':>10}")
        print(f"  {'-'*45}")
        print(f"  {low_turn['name']:<14} {low_turn['left']:>8,} {low_turn['total']:>8,} {low_turn['rate']:>9.1f}%")
        print(f"  {moderate_turn['name']:<14} {moderate_turn['left']:>8,} {moderate_turn['total']:>8,} {moderate_turn['rate']:>9.1f}%")
        print(f"  {high_turn['name']:<14} {high_turn['left']:>8,} {high_turn['total']:>8,} {high_turn['rate']:>9.1f}%")
        print(f"  {'─'*45}")
        print(f"  {'📊 Overall':<14} {data['left_company_last_year'].sum():>8,} {len(data):>8,} {overall_rate:>9.1f}%")
        
        # Mental health metrics
        print("\n📊 MENTAL HEALTH METRICS (Average Scores)")
        print("-"*60)
        
        def get_mental_stats(df):
            return {
                'stress': df['stress_level_self_report'].mean().round(1),
                'anxiety': df['anxiety_score'].mean().round(1),
                'depression': df['depression_score'].mean().round(1),
                'sleep': df['sleep_quality_score'].mean().round(1),
                'job_satisfaction': df['job_satisfaction_score'].mean().round(1)
            }
        
        low_mental = get_mental_stats(burnout_risk_low)
        moderate_mental = get_mental_stats(burnout_risk_moderate)
        high_mental = get_mental_stats(burnout_risk_high)
        
        print(f"  {'Metric':<18} {'🟢 Low':>12} {'🟡 Moderate':>14} {'🔴 High':>12}")
        print(f"  {'-'*58}")
        print(f"  {'⚡ Stress':<18} {low_mental['stress']:>11.1f} {moderate_mental['stress']:>13.1f} {high_mental['stress']:>11.1f}")
        print(f"  {'😰 Anxiety':<18} {low_mental['anxiety']:>11.1f} {moderate_mental['anxiety']:>13.1f} {high_mental['anxiety']:>11.1f}")
        print(f"  {'😔 Depression':<18} {low_mental['depression']:>11.1f} {moderate_mental['depression']:>13.1f} {high_mental['depression']:>11.1f}")
        print(f"  {'😴 Sleep Quality':<18} {low_mental['sleep']:>11.1f} {moderate_mental['sleep']:>13.1f} {high_mental['sleep']:>11.1f}")
        print(f"  {'💼 Job Satisfaction':<18} {low_mental['job_satisfaction']:>11.1f} {moderate_mental['job_satisfaction']:>13.1f} {high_mental['job_satisfaction']:>11.1f}")
        
        # Key insights
        print("\n" + "="*70)
        print("💡 KEY INSIGHTS")
        print("="*70)
        
        hours_diff = high_hours['mean'] - low_hours['mean']
        support_diff = low_support['mean'] - high_support['mean']
        turnover_diff = high_turn['rate'] - low_turn['rate']
        stress_diff = high_mental['stress'] - low_mental['stress']
        
        print(f"""
  📈 Population Insights:
     • Most employees are in Moderate Risk category ({moderate_count/total*100:.1f}%)
     • Only {low_count/total*100:.1f}% of workforce has Low burnout risk

  ⏰ Work Hours Impact:
     • High-risk employees work {hours_diff:.1f} hours MORE than low-risk
     • Low-risk average: {low_hours['mean']} hrs/week (healthy range)
     • High-risk average: {high_hours['mean']} hrs/week (critical range)

  🤝 Social Protection:
     • Low-risk employees have {support_diff:.1f} points HIGHER social support
     • Strong social support = Lower burnout risk

  🚪 Turnout Risk:
     • High-risk employees are {turnover_diff:.1f}% MORE likely to leave
     • Retention improves significantly with lower burnout risk

  💊 Mental Health Correlation:
     • Stress levels: High-risk is {stress_diff:.1f} points higher than low-risk
     • Anxiety and depression show similar patterns
     • Better mental health = Lower burnout risk
""")
        
        # Recommendations
        print("="*70)
        print("🎯 RECOMMENDATIONS BY RISK LEVEL")
        print("="*70)
        
        print(f"""
  🟢 LOW RISK (Maintain & Share):
     • Keep current work-life balance (target: ≤{low_hours['mean']} hrs/week)
     • Act as mentors for moderate-risk employees
     • Share best practices across teams

  🟡 MODERATE RISK (Targeted Intervention):
     • Reduce work hours from {moderate_hours['mean']} to ≤45 hrs/week
     • Improve social support from {moderate_support['mean']} to ≥3.5
     • Weekly stress management sessions
     • Flexible work arrangements

  🔴 HIGH RISK (Urgent Action):
     • Immediate workload reduction (from {high_hours['mean']} to ≤50 hrs/week)
     • Mandatory wellness program participation
     • Increase social support from {high_support['mean']} to ≥3.0
     • One-on-one counseling sessions
     • Job responsibility review
""")
        
        print("="*70)
        print("✅ Analysis Complete")
        print("="*70)
        
        # Restore stdout
        sys.stdout = original_stdout
    
    print(f"\n✅ Report successfully saved to: {filename}")

# Run the function
save_analysis_to_file(data, 'burnout_analysis_report.txt')


✅ Report successfully saved to: burnout_analysis_report.txt


### Correlation

In [33]:
# Convert burnout_risk to numeric
burnout_mapping = {"Low": 0, "Moderate": 1, "High": 2}
data["burnout_risk_numeric"] = data["burnout_risk"].map(burnout_mapping)

# Select only numerical columns for correlation
numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Calculate correlation matrix
correlation = data[numerical_cols].corr()

# Get correlation with burnout_risk_numeric
burnout_corr = correlation["burnout_risk_numeric"].sort_values(ascending=False)

In [34]:
# Create console
console = Console()

# Create a clean table - METHOD 1: Direct printing
print("\n" + "="*70)
print("📊 CORRELATION WITH BURNOUT RISK")
print("="*70)
print(f"{'No.':<4} {'Feature':<35} {'Correlation':<15} {'Strength':<15} {'Direction':<10}")
print("-"*70)

for idx, (feature, corr) in enumerate(burnout_corr.items(), 1):
    if feature == "burnout_risk_numeric":
        continue
    
    # Determine strength
    abs_corr = abs(corr)
    if abs_corr >= 0.5:
        strength = "Very Strong"
    elif abs_corr >= 0.3:
        strength = "Moderate"
    elif abs_corr >= 0.1:
        strength = "Weak"
    else:
        strength = "Negligible"
    
    # Determine direction
    direction = "Positive" if corr > 0 else "Negative" if corr < 0 else "Zero"
    
    # Clean feature name
    feature_name = feature.replace("_", " ").title()
    
    print(f"{idx:<4} {feature_name:<35} {corr:<15.4f} {strength:<15} {direction:<10}")

print("="*70)

# METHOD 2: Rich table (clean and beautiful)
table = Table(
    title="\n[bold cyan]📈 Burnout Risk Correlation Analysis[/bold cyan]",
    box=SIMPLE,
    show_header=True,
    header_style="bold magenta"
)

table.add_column("#", style="dim", width=4)
table.add_column("Feature", style="blue", width=35)
table.add_column("Correlation", style="yellow", justify="right", width=12)
table.add_column("Strength", style="green", width=12)
table.add_column("Direction", style="white", width=10)

for idx, (feature, corr) in enumerate(burnout_corr.items(), 1):
    if feature == "burnout_risk_numeric":
        continue
    
    abs_corr = abs(corr)
    if abs_corr >= 0.5:
        strength = "🔴 Very Strong"
        strength_style = "red"
    elif abs_corr >= 0.3:
        strength = "🟠 Moderate"
        strength_style = "yellow"
    elif abs_corr >= 0.1:
        strength = "🟡 Weak"
        strength_style = "yellow"
    else:
        strength = "⚪ Negligible"
        strength_style = "white"
    
    direction = "➕ Positive" if corr > 0 else "➖ Negative" if corr < 0 else "⚫ Zero"
    feature_name = feature.replace("_", " ").title()
    
    table.add_row(
        str(idx),
        feature_name,
        f"{corr:.4f}",
        f"[{strength_style}]{strength}[/{strength_style}]",
        direction
    )

console.print(table)


📊 CORRELATION WITH BURNOUT RISK
No.  Feature                             Correlation     Strength        Direction 
----------------------------------------------------------------------
2    Hours Worked Per Week               0.6078          Very Strong     Positive  
3    Stress Level Self Report            0.5102          Very Strong     Positive  
4    Anxiety Score                       0.5057          Very Strong     Positive  
5    Depression Score                    0.4428          Moderate        Positive  
6    Left Company Last Year              0.3525          Moderate        Positive  
7    Presenteeism Days Per Month         0.2994          Weak            Positive  
8    Years In Company                    0.0069          Negligible      Positive  
9    Employee Id                         0.0014          Negligible      Positive  
10   Social Support Score                0.0004          Negligible      Positive  
11   Age                                 -0.0107        

                                                                                         
                          📈 Burnout Risk Correlation Analysis                           
                                                                                         
  #      Feature                                Correlation   Strength       Direction   
 ─────────────────────────────────────────────────────────────────────────────────────── 
  2      Hours Worked Per Week                       0.6078   🔴 Very        ➕          
                                                              Strong         Positive    
  3      Stress Level Self Report                    0.5102   🔴 Very        ➕          
                                                              Strong         Positive    
  4      Anxiety Score                               0.5057   🔴 Very        ➕          
                                                              Strong         Positive    
  5      Depression Score                            0.4428   🟠 Moderate    ➕          
                                                                             Positive    
  6      Left Company Last Year                      0.3525   🟠 Moderate    ➕          
                                                                             Positive    
  7      Presenteeism Days Per Month                 0.2994   🟡 Weak        ➕          
                                                                             Positive    
  8      Years In Company                            0.0069   ⚪             ➕          
                                                              Negligible     Positive    
  9      Employee Id                                 0.0014   ⚪             ➕          
                                                              Negligible     Positive    
  10     Social Support Score                        0.0004   ⚪             ➕          
                                                              Negligible     Positive    
  11     Age                                        -0.0107   ⚪             ➖          
                                                              Negligible     Negative    
  12     Work Life Balance Score                    -0.1240   🟡 Weak        ➖          
                                                                             Negative    
  13     Job Satisfaction Score                     -0.3496   🟠 Moderate    ➖          
                                                                             Negative    
  14     Sleep Quality Score                        -0.3889   🟠 Moderate    ➖          
                                                                             Negative   

### model

In [35]:
# 7 features ( with more correlation)
selected_features = [
    'hours_worked_per_week',
    'stress_level_self_report', 
    'anxiety_score',
    'depression_score',
    'left_company_last_year',
    'job_satisfaction_score',
    'sleep_quality_score'
]
# Target
X = data[selected_features]
y = data['burnout_risk']  # Low, Moderate, High

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
# Model 1: Random Forest
print("-"*30)
print("🎯 RANDOM FOREST CLASSIFIER")
print("-"*30)

rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print(f"\nAccuracy: {rf.score(X_test_scaled, y_test):.4f}")
print(f"AUC Score (One-vs-Rest): {roc_auc_score(y_test, rf.predict_proba(X_test_scaled), multi_class='ovr'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

------------------------------
🎯 RANDOM FOREST CLASSIFIER
------------------------------

Accuracy: 0.9895
AUC Score (One-vs-Rest): 0.9993

Classification Report:
              precision    recall  f1-score   support

        High       0.94      0.99      0.97       680
         Low       1.00      0.99      0.99      1095
    Moderate       0.99      0.99      0.99      4225

    accuracy                           0.99      6000
   macro avg       0.98      0.99      0.98      6000
weighted avg       0.99      0.99      0.99      6000



In [37]:
# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Convert to percentage and add a percentage column
importance_df['Importance_Percent'] = (importance_df['Importance'] * 100).round(2)

print("\n📊 Feature Importance (Percentage):")
# Display both decimal and percentage formats
for idx, row in importance_df.iterrows():
    print(f"{row['Feature']:<20} {row['Importance']:.4f} ({row['Importance_Percent']}%)")


📊 Feature Importance (Percentage):
hours_worked_per_week 0.4107 (41.07%)
job_satisfaction_score 0.2876 (28.76%)
anxiety_score        0.1308 (13.08%)
stress_level_self_report 0.0840 (8.4%)
left_company_last_year 0.0394 (3.94%)
depression_score     0.0364 (3.64%)
sleep_quality_score  0.0111 (1.11%)


In [38]:
# Load new dataset (5000 employees)
test_data = pd.read_csv("employees_test.csv")


X_new = test_data[selected_features]
y_true = test_data['burnout_risk']

X_new_scaled = scaler.transform(X_new)

# Make predictions on new, unseen data
y_pred_new = rf.predict(X_new_scaled)

# Evaluate model performance on test set
print("="*60)
print("🎯 Model performance on unseen data")
print("="*60)
print(f"\n📊 Accuracy: {rf.score(X_new_scaled, y_true):.4f}")
print(f"\n📋 Classification Report:")
print(classification_report(y_true, y_pred_new))

🎯 Model performance on unseen data

📊 Accuracy: 0.9914

📋 Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.99      0.98       660
         Low       1.00      0.99      0.99       929
    Moderate       0.99      0.99      0.99      3411

    accuracy                           0.99      5000
   macro avg       0.99      0.99      0.99      5000
weighted avg       0.99      0.99      0.99      5000



In [39]:
import joblib

# Save model, scaler, and selected features
joblib.dump(rf, 'burnout_risk_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(selected_features, 'selected_features.pkl')

print("✅ Model saved successfully!")

✅ Model saved successfully!


### جایزه 